# ✈️ DelayPredict — 03b_v2 XGBoost (Feature Engineering Experiments)

This notebook extends `03b_xgboost.ipynb` with three feature engineering strategies
suggested by our professor. Each can be toggled independently via the config block.

| Toggle | Strategy | Idea |
|---|---|---|
| `USE_TARGET_ENC_COMBINATIONS` | Target-encode pairwise feature combinations | Captures interaction effects |
| `USE_HASH_VECTORIZER` | Hash-bucket the Route feature | Alternative to TargetEncoder for high cardinality |
| `USE_STRING_COMBINATIONS` | Combine 3 columns into one string, then target-encode | Explicit 3-way interactions |

---
**Input:** `data/raw/airlines_delay.csv`  
**Input (optional):** `data/processed/baseline_metrics.csv`, `data/processed/rf_metrics.csv`  
**Output:** `models/xgb_v2_model.pkl`, `data/processed/xgb_v2_metrics.csv`

## 1. Setup and Imports

In [ ]:
import subprocess
subprocess.run(["pip", "install", "mlflow", "xgboost"], check=True)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, TargetEncoder
from sklearn.feature_extraction import FeatureHasher
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score,
    ConfusionMatrixDisplay, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay,
)
from xgboost import XGBClassifier
import joblib
import mlflow
import mlflow.sklearn

pd.set_option("display.max_columns", None)
RANDOM_STATE = 42


## 2. Feature Engineering Toggles

Set each flag to `True` or `False` to enable/disable the strategy.  
Re-run the notebook from top to bottom after changing any toggle.

In [ ]:
# ── Toggle each feature engineering strategy independently ──────────────────

# Idea 1: Target-encode pairwise combinations (e.g. Airline×Hour, Route×DayOfWeek)
# Captures interaction effects that individual features miss
USE_TARGET_ENC_COMBINATIONS = True

# Idea 2: Hash-bucket the Route feature using FeatureHasher
# Alternative high-cardinality encoding — maps routes to fixed-size buckets
USE_HASH_VECTORIZER = True

# Idea 3: Combine 3 columns into one string and target-encode
# e.g. 'WN_MDW-SFO_7' captures 3-way interactions explicitly
USE_STRING_COMBINATIONS = True

print(f"USE_TARGET_ENC_COMBINATIONS : {USE_TARGET_ENC_COMBINATIONS}")
print(f"USE_HASH_VECTORIZER         : {USE_HASH_VECTORIZER}")
print(f"USE_STRING_COMBINATIONS     : {USE_STRING_COMBINATIONS}")


## 3. Load Raw Data

Always loads from the raw dataset — fully self-contained.

In [ ]:
RAW_PATH = Path("../data/raw/airlines_delay.csv")

if not RAW_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {RAW_PATH}")

df = pd.read_csv(RAW_PATH)

print("Source :", RAW_PATH)
print("Shape  :", df.shape)
display(df.head())


## 4. Base Feature Engineering

Core transformations applied in every run — independent of toggles.

In [ ]:
# ── Base features (always applied) ──────────────────────────────────────────
df["DepartureHour"] = df["Time"] // 60
df["Route"]         = df["AirportFrom"] + "-" + df["AirportTo"]

df = df.drop(columns=["id", "Flight", "Time"])

print("Base columns:", df.columns.tolist())


## 5. Experimental Feature Engineering

Each block runs only if its toggle is enabled.

In [ ]:
# ── Idea 1: Pairwise target-encoding combinations ────────────────────────────
# Each combination encodes a specific interaction as its historical delay rate.
# Example: 'WN_7' captures Southwest's delay rate specifically at 7am.
if USE_TARGET_ENC_COMBINATIONS:
    df["Airline_Hour"]    = df["Airline"].astype(str) + "_" + df["DepartureHour"].astype(str)
    df["Route_DayOfWeek"] = df["Route"].astype(str)   + "_" + df["DayOfWeek"].astype(str)
    df["Airline_Route"]   = df["Airline"].astype(str) + "_" + df["Route"].astype(str)
    print("[Idea 1] Added: Airline_Hour, Route_DayOfWeek, Airline_Route")
else:
    print("[Idea 1] Skipped")

# ── Idea 2: Hash Vectorizer for Route ────────────────────────────────────────
# Maps the ~2000 unique routes into a fixed number of hash buckets (n=64).
# Faster and more memory-efficient than OHE, but loses some information.
# Note: TargetEncoder (used in 03a) is more informative — this is a comparison.
if USE_HASH_VECTORIZER:
    hasher = FeatureHasher(n_features=64, input_type="string")
    route_hashed = hasher.transform(df["Route"].apply(lambda x: [x]))
    hash_cols = [f"RouteHash_{i}" for i in range(64)]
    df_hashed = pd.DataFrame(route_hashed.toarray(), columns=hash_cols, index=df.index)
    df = pd.concat([df, df_hashed], axis=1)
    print(f"[Idea 2] Added: {len(hash_cols)} RouteHash columns")
else:
    print("[Idea 2] Skipped")

# ── Idea 3: 3-way string combination + target encoding ───────────────────────
# Combines Airline + Route + DepartureHour into a single string.
# Captures 3-way interactions that pairwise combinations miss.
# Example: 'WN_MDW-SFO_7' = Southwest, Chicago→SF, 7am departure
if USE_STRING_COMBINATIONS:
    df["Airline_Route_Hour"] = (
        df["Airline"].astype(str) + "_" +
        df["Route"].astype(str)   + "_" +
        df["DepartureHour"].astype(str)
    )
    print("[Idea 3] Added: Airline_Route_Hour")
    print(f"          Unique combinations: {df['Airline_Route_Hour'].nunique():,}")
else:
    print("[Idea 3] Skipped")

print(f"\nFinal columns ({len(df.columns)}): {df.columns.tolist()}")


## 6. Features and Target

Feature lists are built dynamically based on which toggles are active.

In [ ]:
TARGET = "Delay"

# ── Base features (always included) ──────────────────────────────────────────
NATIVE_CAT  = ["Airline", "AirportFrom", "AirportTo", "Route", "DayOfWeek"]
NUMERIC     = ["Length", "DepartureHour"]
TARGET_ENC  = []   # populated below based on toggles
HASH_COLS   = []   # populated below based on toggles

# ── Add toggle-specific features ─────────────────────────────────────────────
if USE_TARGET_ENC_COMBINATIONS:
    TARGET_ENC += ["Airline_Hour", "Route_DayOfWeek", "Airline_Route"]

if USE_HASH_VECTORIZER:
    HASH_COLS = [f"RouteHash_{i}" for i in range(64)]

if USE_STRING_COMBINATIONS:
    TARGET_ENC += ["Airline_Route_Hour"]

# ── Build X and y ─────────────────────────────────────────────────────────────
ALL_FEATURES = NATIVE_CAT + NUMERIC + TARGET_ENC + HASH_COLS

X = df[ALL_FEATURES]
y = df[TARGET]

print(f"Native categorical : {NATIVE_CAT}")
print(f"Numeric            : {NUMERIC}")
print(f"Target-encoded     : {TARGET_ENC}")
print(f"Hash columns       : {len(HASH_COLS)} columns" if HASH_COLS else "Hash columns       : none")
print(f"\nTotal features: {len(ALL_FEATURES)}")
print(f"X shape: {X.shape}")


## 7. Train / Test Split

80/20 split with stratify — identical to all other notebooks for fair comparison.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train: {X_train.shape}  |  delay rate: {y_train.mean():.3f}")
print(f"Test : {X_test.shape}   |  delay rate: {y_test.mean():.3f}")


## 8. Preprocessing Pipeline

Three transformer groups, assembled dynamically based on active toggles:
- **Native categoricals** → cast to `pandas.Categorical` for XGBoost
- **Target-encoded features** → replaced by historical delay rate (fitted on train only)
- **Hash columns** → already numeric, passed through unchanged

In [ ]:
# ── Build transformer list dynamically ───────────────────────────────────────
transformers = []

# Native categoricals: XGBoost learns optimal splits on raw categories
def cast_categoricals(X):
    X = X.copy()
    for col in X.select_dtypes(include="object").columns:
        X[col] = X[col].astype("category")
    return X

transformers.append(
    ("native_cat", FunctionTransformer(cast_categoricals), NATIVE_CAT)
)

# Numeric features: passthrough
transformers.append(
    ("num", "passthrough", NUMERIC)
)

# Target-encoded combinations (only if any are active)
if TARGET_ENC:
    transformers.append(
        ("target_enc",
         TargetEncoder(smooth="auto", random_state=RANDOM_STATE),
         TARGET_ENC)
    )

# Hash columns: already numeric, passthrough
if HASH_COLS:
    transformers.append(
        ("hash_pass", "passthrough", HASH_COLS)
    )

preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")
preprocessor.set_output(transform="pandas")  # preserve category dtype for XGBoost

print("Active transformers:")
for name, _, cols in transformers:
    n = len(cols) if isinstance(cols, list) else cols
    print(f"  {name:<15}: {n} features")


## 9. XGBoost Model

Same hyperparameters as `03b_xgboost.ipynb` — only the features change.  
`enable_categorical=True` and `tree_method='hist'` enable native categorical splits.

In [ ]:
xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=10,
        tree_method="hist",        # required for native categorical support
        enable_categorical=True,   # XGBoost learns optimal category splits
        eval_metric="logloss",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )),
])

xgb_pipeline.fit(X_train, y_train)
print("XGBoost trained successfully.")


## 10. Evaluate

Evaluate on the held-out test set.

In [ ]:
# Evaluation metrics — see comments for interpretation
# Accuracy  : share of correct predictions overall
# Precision : of all predicted delays, how many were actual delays
# Recall    : of all actual delays, how many did we catch
# F1        : harmonic mean of Precision and Recall
# ROC-AUC   : ability to rank delayed flights above non-delayed ones (0.5=random, 1.0=perfect)

y_pred_xgb  = xgb_pipeline.predict(X_test)
y_proba_xgb = xgb_pipeline.predict_proba(X_test)[:, 1]

xgb_metrics = {
    "Accuracy" : accuracy_score(y_test, y_pred_xgb),
    "Precision": precision_score(y_test, y_pred_xgb, zero_division=0),
    "Recall"   : recall_score(y_test, y_pred_xgb, zero_division=0),
    "F1"       : f1_score(y_test, y_pred_xgb, zero_division=0),
    "ROC-AUC"  : roc_auc_score(y_test, y_proba_xgb),
}

print("XGBoost v2 metrics:")
for k, v in xgb_metrics.items():
    print(f"  {k:<10}: {v:.4f}")

print()
print(classification_report(y_test, y_pred_xgb,
      target_names=["No Delay", "Delay"], zero_division=0))


## 11. Comparison Against All Models

Load prior metrics and append this run for a full side-by-side view.

In [ ]:
BASELINE_METRICS_PATH = Path("../data/processed/baseline_metrics.csv")
RF_METRICS_PATH       = Path("../data/processed/rf_metrics.csv")
XGB_METRICS_PATH      = Path("../data/processed/xgb_metrics.csv")

xgb_v2_row = {"model": "XGBoost_v2", **xgb_metrics}

prior_frames = []
for path, label in [
    (BASELINE_METRICS_PATH, "baseline_metrics"),
    (RF_METRICS_PATH,       "rf_metrics"),
    (XGB_METRICS_PATH,      "xgb_metrics"),
]:
    if path.exists():
        prior_frames.append(pd.read_csv(path))
    else:
        print(f"{label}.csv not found — skipping.")

comparison_df = pd.concat(
    prior_frames + [pd.DataFrame([xgb_v2_row])],
    ignore_index=True
)

display(comparison_df.round(4))


## 12. Confusion Matrix

- **False Negatives (bottom-left):** predicted No Delay, actually Delay ← the costly error
- **True Positives (bottom-right):** correctly predicted Delay

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_xgb,
    display_labels=["No Delay", "Delay"],
    cmap="Blues", ax=ax,
)
ax.set_title("Confusion Matrix — XGBoost v2")
plt.tight_layout()
plt.show()


## 13. ROC Curve and Precision-Recall Curve

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

RocCurveDisplay.from_predictions(y_test, y_proba_xgb, ax=ax1, name="XGBoost v2")
ax1.plot([0, 1], [0, 1], "k--", label="Random (AUC = 0.5)")
ax1.set_title("ROC Curve — XGBoost v2")
ax1.legend()

PrecisionRecallDisplay.from_predictions(y_test, y_proba_xgb, ax=ax2, name="XGBoost v2")
ax2.axhline(y_test.mean(), color="k", linestyle="--",
            label=f"No-skill baseline ({y_test.mean():.2f})")
ax2.set_title("Precision-Recall Curve — XGBoost v2")
ax2.legend()

plt.tight_layout()
plt.show()


## 14. Feature Importance

Shows which features XGBoost used most for splits.  
Interaction features (Idea 1 & 3) should rank higher than base features if they add signal.

In [ ]:
# Reconstruct feature names in ColumnTransformer output order
out_names = NATIVE_CAT + NUMERIC + TARGET_ENC + HASH_COLS
importances = xgb_pipeline.named_steps["classifier"].feature_importances_

importance_df = (
    pd.DataFrame({"Feature": out_names, "Importance": importances})
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

# Show top 20 — hash columns are many, cap display
display(importance_df.head(20))

# Bar chart: top 15 non-hash features for readability
plot_df = (
    importance_df[~importance_df["Feature"].str.startswith("RouteHash_")]
    .head(15)
    .sort_values("Importance")
)

plt.figure(figsize=(9, 5))
plt.barh(plot_df["Feature"], plot_df["Importance"], color="steelblue")
plt.title("Top 15 Feature Importances — XGBoost v2 (excl. hash columns)")
plt.xlabel("Feature Importance")
plt.tight_layout()
plt.show()


## 15. Predicted Probability Distribution

Better separation between the two histograms = better model confidence.  
Compare to `03b_xgboost.ipynb` to see if new features helped.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(y_proba_xgb[y_test == 0], bins=50, alpha=0.6,
         label="No Delay (actual)", color="steelblue")
plt.hist(y_proba_xgb[y_test == 1], bins=50, alpha=0.6,
         label="Delay (actual)", color="tomato")
plt.axvline(0.5, color="black", linestyle="--", label="Threshold 0.5")
plt.title("Predicted Probability Distribution — XGBoost v2")
plt.xlabel("P(Delay)")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()


## 16. Save Model and Metrics

In [ ]:
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / "xgb_v2_model.pkl"
joblib.dump(xgb_pipeline, MODEL_PATH)
print(f"Model saved   : {MODEL_PATH}")

OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

xgb_v2_metrics_df = pd.DataFrame([{"model": "XGBoost_v2", **xgb_metrics}])
XGB_V2_METRICS_PATH = OUTPUT_DIR / "xgb_v2_metrics.csv"
xgb_v2_metrics_df.to_csv(XGB_V2_METRICS_PATH, index=False)
print(f"Metrics saved : {XGB_V2_METRICS_PATH}")


## 17. MLflow Logging

Logs active toggles as parameters — so every run is fully reproducible in the UI.

In [ ]:
mlflow.set_tracking_uri("file:///app/notebooks/mlruns")

mlflow.set_experiment("DelayPredict")

with mlflow.start_run(run_name="XGBoost_v2_FeatureExperiment"):

    xgb_clf = xgb_pipeline.named_steps["classifier"]

    # Log toggles so each run is reproducible
    mlflow.log_params({
        "model"                       : "XGBoost_v2",
        "USE_TARGET_ENC_COMBINATIONS" : USE_TARGET_ENC_COMBINATIONS,
        "USE_HASH_VECTORIZER"         : USE_HASH_VECTORIZER,
        "USE_STRING_COMBINATIONS"     : USE_STRING_COMBINATIONS,
        "n_estimators"                : xgb_clf.n_estimators,
        "learning_rate"               : xgb_clf.learning_rate,
        "max_depth"                   : xgb_clf.max_depth,
        "total_features"              : len(ALL_FEATURES),
        "random_state"                : RANDOM_STATE,
    })

    mlflow.log_metrics({
        "accuracy" : xgb_metrics["Accuracy"],
        "precision": xgb_metrics["Precision"],
        "recall"   : xgb_metrics["Recall"],
        "f1"       : xgb_metrics["F1"],
        "roc_auc"  : xgb_metrics["ROC-AUC"],
    })

    mlflow.sklearn.log_model(xgb_pipeline, artifact_path="model")

    print(f"MLflow run logged — experiment: 'DelayPredict'")
    print(f"Toggles: TEC={USE_TARGET_ENC_COMBINATIONS}, "
          f"HV={USE_HASH_VECTORIZER}, SC={USE_STRING_COMBINATIONS}")


## 18. Summary

In [ ]:
print("XGBOOST v2 RESULTS")
print("=" * 72)
print(f"Train : {X_train.shape[0]:>7,} rows")
print(f"Test  : {X_test.shape[0]:>7,} rows")
print(f"Active features ({len(ALL_FEATURES)}): {ALL_FEATURES}")
print()
print(f"  Target Enc Combinations : {USE_TARGET_ENC_COMBINATIONS}")
print(f"  Hash Vectorizer         : {USE_HASH_VECTORIZER}")
print(f"  String Combinations     : {USE_STRING_COMBINATIONS}")
print()

metric_cols = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
print(f"{'Model':<22}" + "".join(f"{m:>10}" for m in metric_cols))
print("-" * 72)
for _, row in comparison_df.iterrows():
    vals = "".join(f"{row[m]:>10.4f}" for m in metric_cols)
    print(f"{row['model']:<22}{vals}")
